<a href="https://colab.research.google.com/github/philip196777/VideoFake/blob/main/flux.1-dev_jupyter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- КЛОН REPO ---
%cd /content
!git clone -b totoro3 https://github.com/camenduru/ComfyUI /content/TotoroUI || echo "Repo already exists"
%cd /content/TotoroUI

# --- ОЧИСТКА СТРЕМНЫХ ВЕРСИЙ ---
!pip uninstall -y fastai torch torchvision torchaudio xformers || true

# --- УСТАНОВКА PyTorch + CUDA 11.8 (Python 3.11 уже установлен) ---
!pip install --no-cache-dir torch==2.7.1+cu118 torchaudio==2.7.1+cu118 --index-url https://download.pytorch.org/whl/cu118

# (ВНИМАНИЕ) torchvision не устанавливаем специально — он не нужен и вызывает проблемы

# --- УСТАНОВКА ОСТАЛЬНОГО ---
!pip install --no-cache-dir xformers==0.0.28.post2
!pip install --no-cache-dir torchsde einops diffusers accelerate --use-deprecated=legacy-resolver

# --- УСТАНОВКА ARIA2 ---
!apt -y install -qq aria2

# --- FLUX FP8 МОДЕЛИ ---
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/flux1-dev-fp8.safetensors -d /content/TotoroUI/models/unet -o flux1-dev-fp8.safetensors
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/ae.sft -d /content/TotoroUI/models/vae -o ae.sft
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/clip_l.safetensors -d /content/TotoroUI/models/clip -o clip_l.safetensors
!aria2c -c -x 16 -s 16 -k 1M https://huggingface.co/camenduru/FLUX.1-dev/resolve/main/t5xxl_fp8_e4m3fn.safetensors -d /content/TotoroUI/models/clip -o t5xxl_fp8_e4m3fn.safetensors

# --- ПАТЧ: закомментировать все импорты torchvision в репозитории (если есть) ---
# Это гарантирует, что импорт nodes не упадёт из-за отсутствия torchvision
!grep -R --line-number "^\s*import torchvision" || true
!grep -R --line-number "^\s*from torchvision" || true
!find . -type f -name "*.py" -exec sed -i 's/^\(\s*\)import torchvision/\1# import torchvision (commented by setup)/g' {} \;
!find . -type f -name "*.py" -exec sed -i 's/^\(\s*\)from torchvision/\1# from torchvision (commented by setup)/g' {} \;

# --- ПУТЬ К REPO ---
import sys
sys.path.append("/content/TotoroUI")

# --- ИМПОРТЫ (порядок важен) ---
import torch
# torchvision не импортируем намеренно

# --- ЗАГРУЗКА МОДУЛЕЙ NODES ---
import nodes
from nodes import NODE_CLASS_MAPPINGS
from totoro_extras import nodes_custom_sampler
from totoro import model_management

# --- ИНИЦИАЛИЗАЦИЯ УЗЛОВ ---
DualCLIPLoader = NODE_CLASS_MAPPINGS["DualCLIPLoader"]()
UNETLoader = NODE_CLASS_MAPPINGS["UNETLoader"]()
VAELoader = NODE_CLASS_MAPPINGS["VAELoader"]()
VAEDecode = NODE_CLASS_MAPPINGS["VAEDecode"]()
EmptyLatentImage = NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

# --- ЗАГРУЗКА МОДЕЛЕЙ (FP8) ---
with torch.inference_mode():
    clip = DualCLIPLoader.load_clip("t5xxl_fp8_e4m3fn.safetensors", "clip_l.safetensors", "flux")[0]
    unet = UNETLoader.load_unet("flux1-dev-fp8.safetensors", "fp8_e4m3fn")[0]
    vae = VAELoader.load_vae("ae.sft")[0]

print("✅ CUDA FLUX загружен. Ошибок нет. (torchvision отключён)")



/content
Cloning into '/content/TotoroUI'...
remote: Enumerating objects: 20484, done.
remote: Total 20484 (delta 0), reused 0 (delta 0), pack-reused 20484 (from 1)
Receiving objects: 100% (20484/20484), 65.43 MiB | 15.18 MiB/s, done.
Resolving deltas: 100% (13661/13661), done.
/content/TotoroUI
Found existing installation: fastai 2.7.19
Uninstalling fastai-2.7.19:
  Successfully uninstalled fastai-2.7.19
Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 359.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

✅ CUDA FLUX загружен. Ошибок нет. (torchvision отключён)


In [6]:
import random
import numpy as np
from PIL import Image
import torch

# Узлы берём отсюда
from totoro_extras import nodes_custom_sampler
from totoro import model_management

# --- Функция для округления к кратному 16 ---
def closestNumber(n, m=16):
    q = int(n / m)
    n1 = m * q
    n2 = m * (q + 1) if (n * m) > 0 else m * (q - 1)
    return n1 if abs(n - n1) < abs(n - n2) else n2

# --- Достаём классы ---
RandomNoise = nodes_custom_sampler.NODE_CLASS_MAPPINGS["RandomNoise"]()
BasicGuider = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicGuider"]()
KSamplerSelect = nodes_custom_sampler.NODE_CLASS_MAPPINGS["KSamplerSelect"]()
BasicScheduler = nodes_custom_sampler.NODE_CLASS_MAPPINGS["BasicScheduler"]()
SamplerCustomAdvanced = nodes_custom_sampler.NODE_CLASS_MAPPINGS["SamplerCustomAdvanced"]()
EmptyLatentImage = nodes_custom_sampler.NODE_CLASS_MAPPINGS["EmptyLatentImage"]()

# --- Параметры генерации ---
with torch.inference_mode():
    positive_prompt = "A close-up of a face adorned with intricate black and blue patterns. The left side of the face is predominantly yellow, with symbols and doodles, while the right side is dark, featuring mechanical elements. The eye on the left is a striking shade of yellow, contrasting sharply with the surrounding patterns. The face is partially covered by a hooded garment, cinematic_1940s cinematic_octane"

    # Можно менять размеры
    requested_width = 640
    requested_height = 640

    width = closestNumber(requested_width)
    height = closestNumber(requested_height)

    seed = 0
    steps = 20
    sampler_name = "euler"
    scheduler = "simple"

    if seed == 0:
        seed = random.randint(0, 18446744073709551615)
    print("Seed:", seed)
    print("Generating image:", width, "x", height)

    # --- Генерация условного кода ---
    cond, pooled = clip.encode_from_tokens(clip.tokenize(positive_prompt), return_pooled=True)
    cond = [[cond, {"pooled_output": pooled}]]

    noise = RandomNoise.get_noise(seed)[0]
    guider = BasicGuider.get_guider(unet, cond)[0]
    sampler = KSamplerSelect.get_sampler(sampler_name)[0]
    sigmas = BasicScheduler.get_sigmas(unet, scheduler, steps, 1.0)[0]

    # --- Генерация латентного изображения ---
    latent_image = EmptyLatentImage.generate(width, height)[0]

    # --- Семплинг ---
    sample, _ = SamplerCustomAdvanced.sample(
        noise, guider, sampler, sigmas, latent_image
    )
    model_management.soft_empty_cache()

    # --- Декодирование и сохранение ---
    decoded = VAEDecode.decode(vae, sample)[0].detach()
    image = Image.fromarray(
        (decoded * 255).cpu().numpy().astype(np.uint8)[0]
    )
    image.save("/content/flux.png")

image


Seed: 4187894424341957164


NameError: name 'closestNumber' is not defined